Import Libraries

In [7]:
import numpy as np
import time
from datetime import datetime, timedelta

# Define parameters
learning_rate = 0.5
discount_factor = 0.9
exploration_rate = 1.0
exploration_decay = 0.99
min_exploration_rate = 0.01
num_episodes = 200  # Reduced for faster training
max_steps = 100  # Max steps per episode

# State space definition (Subject Difficulty, Time Required, Completion Percentage)
state_space = {
    'subject_difficulty': ['Easy', 'Medium', 'Hard'],  # 3 levels
    'time_required': ['Short', 'Moderate', 'Long'],  # 3 levels
    'completion_percentage': np.linspace(0, 100, 10),  # 10 bins for completion percentage (0-100%)
}

# Study time recommendations in hours
study_times = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0]


Generate Code

In [8]:
# Function to generate time slots based on the user's total time allocation for the day
def generate_time_slots(start_time, end_time, slot_duration):
    time_slots = []
    current_time = start_time
    while current_time + timedelta(hours=slot_duration) <= end_time:
        slot_end = current_time + timedelta(hours=slot_duration)
        time_slots.append(f"{current_time.strftime('%H:%M')} - {slot_end.strftime('%H:%M')}")
        current_time = slot_end
    return time_slots

# Function to discretize days until exam, past score, and completion percentage
def discretize_value(value, bins):
    return min(int(value / (100 / bins)), bins - 1)

# Helper function to convert state to index
def get_state_index(subject_difficulty, days_until_exam, past_score, time_required, completion_percentage):
    diff_idx = state_space['subject_difficulty'].index(subject_difficulty)
    time_required_idx = state_space['time_required'].index(time_required)
    
    # Discretize days_until_exam, past_score, and completion_percentage
    days_idx = discretize_value(days_until_exam, 10)  # 10 bins for days until exam
    score_idx = discretize_value(past_score, 10)  # 10 bins for past score
    completion_idx = discretize_value(completion_percentage, 10)  # 10 bins for completion percentage
    
    # Combine the indices to create a unique index for each state
    return (diff_idx * len(state_space['time_required']) * 10 * 10 * 10 +
            time_required_idx * 10 * 10 * 10 +
            days_idx * 10 * 10 +
            score_idx * 10 +
            completion_idx)

# Helper function to convert action to study time and time slot
def get_action(action_index):
    time_idx = action_index // len(time_slots)
    slot_idx = action_index % len(time_slots)
    return study_times[time_idx], time_slots[slot_idx]


In [9]:
# Q-table initialization
state_space_size = len(state_space['subject_difficulty']) * len(state_space['time_required']) * 10 * 10 * 10

# Ensure time_slots is defined
if 'time_slots' not in globals():
    start_time = datetime.strptime("08:00", '%H:%M')  # Default start time
    end_time = datetime.strptime("20:00", '%H:%M')  # Default end time
    time_slots = generate_time_slots(start_time, end_time, 2.0)  # Default 2-hour slots

action_space_size = len(study_times) * len(time_slots)
q_table = np.zeros((state_space_size, action_space_size))

# Q-learning training function
def train_q_learning():
    global exploration_rate
    start_time = time.time()
    
    for episode in range(num_episodes):
        # Randomly initialize the state for each episode
        subject_difficulty = np.random.choice(state_space['subject_difficulty'])
        days_until_exam = np.random.uniform(1, 30)  # Random value between 1 and 30
        past_score = np.random.uniform(0, 100)  # Random value between 0 and 100
        time_required = np.random.choice(state_space['time_required'])
        completion_percentage = np.random.uniform(0, 100)  # Random completion percentage
        
        state = get_state_index(subject_difficulty, days_until_exam, past_score, time_required, completion_percentage)
        
        for step in range(max_steps):
            # Exploration-exploitation trade-off
            if np.random.uniform(0, 1) < exploration_rate:
                action = np.random.choice(action_space_size)  # Explore
            else:
                action = np.argmax(q_table[state, :])  # Exploit (best action)

            # Take the action and simulate the next state (simplified for now)
            next_state = get_state_index(subject_difficulty, days_until_exam, past_score, time_required, completion_percentage)
            
            # Reward: Simple reward based on lower past score and fewer days left
            reward = (100 - past_score) / 100 + (30 - days_until_exam) / 30 + (100 - completion_percentage) / 100

            # Q-learning update
            q_table[state, action] = q_table[state, action] + learning_rate * (
                reward + discount_factor * np.max(q_table[next_state, :]) - q_table[state, action])

            # Move to the next state
            state = next_state

        # Decay exploration rate
        exploration_rate = max(min_exploration_rate, exploration_rate * exploration_decay)

    total_time = time.time() - start_time
    print(f"Training completed in {total_time:.2f} seconds")


In [10]:
# Track allocated time slots to avoid overlap
allocated_time_slots = set()

# Function to get study schedule recommendation based on the trained Q-table
def get_study_schedule(subjects):
    schedule = {}
    
    for subject in subjects:
        subject_difficulty = subject['subject_difficulty']
        days_until_exam = subject['days_until_exam']
        past_score = subject['past_score']
        time_required = subject['time_required']
        completion_percentage = subject['completion_percentage']
        
        state = get_state_index(subject_difficulty, days_until_exam, past_score, time_required, completion_percentage)
        action = np.argmax(q_table[state, :])
        study_time, time_slot = get_action(action)
        
        # Adjust the time slot based on the recommended study time
        slot_start, slot_end = time_slot.split(' - ')
        slot_start = datetime.strptime(slot_start, '%H:%M')
        slot_end = datetime.strptime(slot_end, '%H:%M')
        
        # Calculate the actual end time based on the recommended study time
        adjusted_slot_end = slot_start + timedelta(hours=study_time)
        
        # Keep it within bounds if exceeds the original slot
        if adjusted_slot_end > slot_end:
            adjusted_slot_end = slot_end
        
        adjusted_time_slot = f"{slot_start.strftime('%H:%M')} - {adjusted_slot_end.strftime('%H:%M')}"
        
        # Check if the time slot is already allocated
        if adjusted_time_slot in allocated_time_slots:
            continue  # Skip if the slot is already taken
        
        # Allocate the time slot to this subject
        allocated_time_slots.add(adjusted_time_slot)
        schedule[subject['name']] = (study_time, adjusted_time_slot)
    
    return schedule


In [14]:
# Track allocated time slots to avoid overlap
allocated_time_slots = set()

# Function to get study schedule recommendation based on the trained Q-table
def get_study_schedule(subjects):
    schedule = {}
    
    for subject in subjects:
        subject_difficulty = subject['subject_difficulty']
        days_until_exam = subject['days_until_exam']
        past_score = subject['past_score']
        time_required = subject['time_required']
        completion_percentage = subject['completion_percentage']
        
        state = get_state_index(subject_difficulty, days_until_exam, past_score, time_required, completion_percentage)
        action = np.argmax(q_table[state, :])
        study_time, time_slot = get_action(action)
        
        # Adjust the time slot based on the recommended study time
        slot_start, slot_end = time_slot.split(' - ')
        slot_start = datetime.strptime(slot_start, '%H:%M')
        slot_end = datetime.strptime(slot_end, '%H:%M')
        
        # Calculate the actual end time based on the recommended study time
        adjusted_slot_end = slot_start + timedelta(hours=study_time)
        
        # If the adjusted end time exceeds the original slot, keep it within bounds
        if adjusted_slot_end > slot_end:
            adjusted_slot_end = slot_end
        
        adjusted_time_slot = f"{slot_start.strftime('%H:%M')} - {adjusted_slot_end.strftime('%H:%M')}"
        
        # Check if the time slot is already allocated, and if it is, find the next available slot
        while adjusted_time_slot in allocated_time_slots:
            # If the slot is already taken, move to the next available one
            time_idx = time_slots.index(time_slot) + 1
            if time_idx >= len(time_slots):
                print(f"Unable to allocate time slot for {subject['name']} due to conflicts.")
                continue  # Skip if no available slots

            time_slot = time_slots[time_idx]
            slot_start, slot_end = time_slot.split(' - ')
            slot_start = datetime.strptime(slot_start, '%H:%M')
            slot_end = datetime.strptime(slot_end, '%H:%M')
            adjusted_slot_end = slot_start + timedelta(hours=study_time)
            if adjusted_slot_end > slot_end:
                adjusted_slot_end = slot_end
            adjusted_time_slot = f"{slot_start.strftime('%H:%M')} - {adjusted_slot_end.strftime('%H:%M')}"
        
        # Allocate the time slot to this subject
        allocated_time_slots.add(adjusted_time_slot)
        schedule[subject['name']] = (study_time, adjusted_time_slot)
    
    return schedule



In [15]:
# Function to input subjects dynamically
def input_subjects():
    subjects = []
    num_subjects = int(input("Enter the number of subjects: "))
    
    for i in range(num_subjects):
        print(f"\nEnter details for Subject {i + 1}:")
        name = input("Subject Name: ")
        difficulty = input("Subject Difficulty (Easy/Medium/Hard): ")
        days_until_exam = float(input("Days Until Exam: "))
        past_score = float(input("Past Score (percentage): "))
        time_required = input("Time Required (Short/Moderate/Long): ")
        completion_percentage = float(input("Completion Percentage: "))
        
        subject = {
            'name': name,
            'subject_difficulty': difficulty,
            'days_until_exam': days_until_exam,
            'past_score': past_score,
            'time_required': time_required,
            'completion_percentage': completion_percentage
        }
        
        subjects.append(subject)
    
    return subjects

# Add, delete or modify subjects
def modify_subjects(subjects):
    while True:
        print("\n--- Modify Subjects Menu ---")
        print("1. Add a subject")
        print("2. Delete a subject")
        print("3. Modify completion percentage")
        print("4. Exit")
        choice = int(input("Enter your choice: "))

        if choice == 1:  # Add a subject
            new_subject = input_subjects()[0]
            subjects.append(new_subject)
            print(f"Added new subject: {new_subject['name']}")
        
        elif choice == 2:  # Delete a subject
            subject_name = input("Enter the name of the subject to delete: ")
            subjects = [s for s in subjects if s['name'] != subject_name]
            print(f"Deleted subject: {subject_name}")
        
        elif choice == 3:  # Modify completion percentage
            subject_name = input("Enter the name of the subject to modify: ")
            for subject in subjects:
                if subject['name'] == subject_name:
                    new_completion = float(input(f"Enter new completion percentage for {subject_name}: "))
                    subject['completion_percentage'] = new_completion
                    print(f"Updated completion percentage for {subject_name} to {new_completion}%")
                    break
            else:
                print(f"Subject {subject_name} not found.")
        
        elif choice == 4:
            break
        
        else:
            print("Invalid choice. Please try again.")
    
    return subjects


In [16]:
# Example subjects for testing
test_subjects = [
    {
        'name': 'Mathematics',
        'subject_difficulty': 'Hard',
        'days_until_exam': 15,
        'past_score': 65,
        'time_required': 'Long',
        'completion_percentage': 40.0
    },
    {
        'name': 'Physics',
        'subject_difficulty': 'Medium',
        'days_until_exam': 10,
        'past_score': 75,
        'time_required': 'Moderate',
        'completion_percentage': 50.0
    },
    {
        'name': 'Chemistry',
        'subject_difficulty': 'Easy',
        'days_until_exam': 20,
        'past_score': 85,
        'time_required': 'Short',
        'completion_percentage': 30.0
    }
]

# Example time for study (8 AM to 8 PM)
start_time_str = "08:00"
end_time_str = "20:00"

# Ensure time_slots is defined
if 'time_slots' not in globals():
    # Convert strings to datetime objects for generating time slots
    start_time = datetime.strptime(start_time_str, '%H:%M')
    end_time = datetime.strptime(end_time_str, '%H:%M')

    # Generate time slots of 2 hours each
    time_slots = generate_time_slots(start_time, end_time, 2.0)

# Train the Q-learning model
train_q_learning()

# Get the study schedule for the input test subjects
schedule = get_study_schedule(test_subjects)
print("Generated Study Schedule:", schedule)


Training completed in 0.40 seconds
Generated Study Schedule: {'Mathematics': (0.5, '08:00 - 08:30'), 'Physics': (0.5, '10:00 - 10:30'), 'Chemistry': (0.5, '12:00 - 12:30')}


In [17]:
import pickle

# Save the Q-table to a pickle file
with open('q_table_model.pkl', 'wb') as file:
    pickle.dump(q_table, file)
    
print("Model saved to q_table_model.pkl")


Model saved to q_table_model.pkl
